# Семинар 3. Обзор А/Б-тест фреймворков. tea-tasting, HypEx, Growthbook.

In [ ]:
!pip install tea-tasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00


In [ ]:
import random
import pandas as pd
import numpy as np
import polars as pl
import tea_tasting as tt

pd.set_option('future.no_silent_downcasting', True)

In [ ]:
!python --version

Python 3.12.12


# [tea_tasting](https://github.com/e10v/tea-tasting) (потому что созвучно t-testing)

tea-tasting is a Python package for the statistical analysis of A/B tests featuring:

- Student's t-test, Z-test, bootstrap, and quantile metrics out of the box.
- Extensible API that lets you define and use statistical tests of your choice.
- Delta method for ratio metrics.
- Variance reduction using CUPED/CUPAC, which can be combined with the Delta method for ratio metrics.
- Confidence intervals for both absolute and percentage changes.
- Checks for sample-ratio mismatches.
- Power analysis.
- Simulated experiments, including A/A tests.
- Multiple hypothesis testing (family-wise error rate and false discovery rate).


## Генерация датасета - синтетика «интернет-магазин»


In [ ]:
import tea_tasting as tt

# по умолчанию в колоночном бинарном формате Arrow, оптимизированном для скорости и памяти
data = tt.make_users_data(seed=42, n_users=10_000)

# user: The unique identifier for each user.
# variant: The specific variant (e.g., 0 or 1) assigned to each user in the A/B test.
# sessions: The total number of user's sessions.
# orders: The total number of user's orders.
# revenue: The total revenue generated by the user.

data.slice(0, 5)

pyarrow.Table
user: int64
variant: int64
sessions: int64
orders: int64
revenue: double
----
user: [[0,1,2,3,4]]
variant: [[1,0,1,1,0]]
sessions: [[3,1,1,1,3]]
orders: [[0,1,0,1,1]]
revenue: [[0,20.02,0,11.53,10.68]]

In [ ]:
# для CUPED - генерирует сразу ковариаты
# (обычно берут это уже метрику в прошлом периоде)
data = tt.make_users_data(seed=42, covariates=True)
data

pyarrow.Table
user: int64
variant: int64
sessions: int64
orders: int64
revenue: double
sessions_covariate: int64
orders_covariate: int64
revenue_covariate: double
----
user: [[0,1,2,3,4,...,3995,3996,3997,3998,3999]]
variant: [[1,0,1,1,0,...,0,0,0,0,0]]
sessions: [[2,2,2,2,1,...,2,2,3,1,5]]
orders: [[1,1,1,1,1,...,0,0,0,0,2]]
revenue: [[9.17,6.43,7.94,15.93,7.14,...,0,0,0,0,17.16]]
sessions_covariate: [[3,4,4,1,1,...,1,3,2,1,5]]
orders_covariate: [[2,1,2,0,1,...,0,1,0,0,0]]
revenue_covariate: [[19.19,2.77,22.57,0,13.68,...,0,13.52,0,0,0]]

In [ ]:
# можно и пандас получить
data = tt.make_users_data(seed=42, return_type="pandas")
pd.concat([data.head(), data.tail()])

,user,variant,sessions,orders,revenue
0,0,1,2,1,9.17
1,1,0,2,1,6.43
2,2,1,2,1,7.94
3,3,1,2,1,15.93
4,4,0,1,1,7.14
3995,3995,0,2,0,0.00
3996,3996,0,2,0,0.00
3997,3997,0,3,0,0.00
3998,3998,0,1,0,0.00
3999,3999,0,5,2,17.16


# Задаем метрики + критерии

! При определении метрики мы сразу задаем по умолчанию критерии, которые будут использованы для нее

- Mean = t-test (scipy)
- RatioOfMeans = z-test на отношении средних из двух разных метрик, с пересчетом дисперсий с помощью дельта-метода
- Bootstrap (scipy.stats.bootstrap)


In [ ]:
experiment = tt.Experiment(
    t_test_revenue_per_user=tt.Mean("revenue"),
    z_test_orders_per_session=tt.RatioOfMeans("orders", "sessions"),
    bootstrap_orders_per_user=tt.Bootstrap("orders", np.mean, random_state=42)
)

# Один тест

- rel_effect_size = Относительный эффект метрики = (метрики treatment − метрики control) / метрики control
- [rel_effect_size_ci_lower, rel_effect_size_ci_upper] - Это 95% доверительный интервал относительного эффекта


In [ ]:
result = experiment.analyze(data)
print(result)

                   metric control treatment rel_effect_size rel_effect_size_ci   pvalue
  t_test_revenue_per_user    4.99      5.75             15%        [6.8%, 24%] 2.54e-04
z_test_orders_per_session   0.246     0.285             16%        [8.9%, 23%] 2.89e-06
bootstrap_orders_per_user   0.494     0.577             17%        [9.5%, 25%]        -


# Функциональность А/А-теста в tea-testing

- как разбивает на группы: генерит группу через `np.random.Generator.binomia(n=1, p=ratio / (1 + ratio), size=data.num_rows)`

https://github.com/e10v/tea-tasting/blob/61cbbd6ea763d661cc356e2bbfa4ed085acb9ee9/src/tea_tasting/experiment.py#L549

- metric: Metric name.
- control: Mean or ratio of means in the control variant.
- treatment: Mean or ratio of means in the treatment variant.
- effect_size: Absolute effect size. Difference between two means.
- effect_size_ci_lower: Lower bound of the absolute effect size confidence interval.
- effect_size_ci_upper: Upper bound of the absolute effect size confidence interval.
- rel_effect_size: Relative effect size. Difference between two means, divided by the control mean.
- rel_effect_size_ci_lower: Lower bound of the relative effect size confidence interval.
- rel_effect_size_ci_upper: Upper bound of the relative effect size confidence interval.
- pvalue: P-value
- statistic: Statistic (standardized effect size).


In [ ]:
results = experiment.simulate(data, 100, seed=42) # data должна быть PyArrow
results_data = results.to_polars()
results_data.select(
    "metric",
    "control",
    "treatment",
    "rel_effect_size",
    "rel_effect_size_ci_lower",
    "rel_effect_size_ci_upper",
    "pvalue",
)

metric,control,treatment,rel_effect_size,rel_effect_size_ci_lower,rel_effect_size_ci_upper,pvalue
str,f64,f64,f64,f64,f64,f64
"""t_test_revenue_per_user""",5.431314,5.302713,-0.023678,-0.095049,0.053323,0.536313
"""z_test_orders_per_session""",0.264603,0.266323,0.006501,-0.053279,0.070056,0.835668
"""bootstrap_orders_per_user""",0.538644,0.531893,-0.012533,-0.073586,0.053627,null
"""t_test_revenue_per_user""",5.649205,5.090696,-0.098865,-0.164709,-0.027831,0.007314
"""z_test_orders_per_session""",0.277872,0.253091,-0.089181,-0.143255,-0.031695,0.002798
…,…,…,…,…,…,…
"""z_test_orders_per_session""",0.261069,0.269757,0.033276,-0.028146,0.098581,0.294862
"""bootstrap_orders_per_user""",0.52757,0.54288,0.029019,-0.034851,0.098463,null
"""t_test_revenue_per_user""",5.395416,5.339379,-0.010386,-0.08284,0.067792,0.78776


## Аггрегируем результат АА для разных размеров эффекта

In [ ]:
def null_rejected(
     results_data: pl.DataFrame,
     alphas: tuple[float, ...] = (0.01, 0.03, 0.05),
) -> pl.DataFrame:
    return results_data.group_by("metric", maintain_order=True).agg(
         pl.col("pvalue").le(alpha).mean().alias(f"null_rejected_{alpha}")
         for alpha in alphas
         )

null_rejected(results_data)

metric,null_rejected_0.01,null_rejected_0.03,null_rejected_0.05
str,f64,f64,f64
"""t_test_revenue_per_user""",0.0,0.0,0.0
"""z_test_orders_per_session""",0.0,0.02,0.04
"""bootstrap_orders_per_user""",null,null,null


# Механика анализа MDE/мощности

при каком минимальном эффекте (или при каком n, или при какой power — зависит от параметра) классический t/z-тест будет иметь заданную мощность

In [ ]:
# для бутстрепа не подбирает MDE, только для t/z-теста
power_result = experiment.solve_power(data)
power_result

metric,power,effect_size,rel_effect_size,n_obs
t_test_revenue_per_user,80%,0.888,16%,4000
z_test_orders_per_session,80%,0.0368,13%,4000
